# MAPED on one CUDA GPU

Keep all seven inputs encoded on GPU0. MAPED computes in float32 once; QuantEM.GPU automatically calibrates and packs the complete merged output as scaled uint16. Viewing does not require saving or reopening a file. Set `MAPED_DATA_DIR` to the seven-input directory and optionally `MAPED_OUTPUT` for a later export. The same processing call works with Torch MPS on Phil.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from pathlib import Path
from time import perf_counter
import torch
from quantem.diffraction import MAPEDTorch

try:
    SESSION = Path(os.environ["MAPED_DATA_DIR"]).expanduser().resolve()
except KeyError as exc:
    raise RuntimeError("Set MAPED_DATA_DIR to the directory containing seven *_master.h5 files.") from exc
FILES = sorted(SESSION.glob("*_master.h5"))
if len(FILES) != 7:
    raise ValueError(f"Expected seven *_master.h5 files in {SESSION}, found {len(FILES)}.")
OUTPUT = Path(os.environ.get("MAPED_OUTPUT", "maped-output/merged_master.h5")).expanduser().resolve()
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
print(torch.cuda.get_device_name(0), "|", len(FILES), "tilts")


In [ ]:
started = perf_counter()
maped = MAPEDTorch.from_files(FILES, device="cuda:0")
torch.cuda.synchronize()
sources = maped.datasets.sources
assert len(sources) == len(FILES) == 7
assert all(source.representation.value == "encoded" for source in sources)
assert all(source.metadata["source_read_passes"] == 1 for source in sources)
print(f"Encoded resident ready: {sum(x.resident_bytes for x in sources) / 2**30:.2f} GiB in {perf_counter() - started:.2f} s")


In [ ]:
started = perf_counter()
maped.preprocess(plot_summary=False)
maped.diffraction_origin(sigma=1, plot_origins=False)
maped.diffraction_align(edge_blend=2, plot_aligned=False)
maped.real_space_align(
    num_iter=20, hanning_filter=True, padding=2, edge_blend=5,
    pad_val="median", shift_method="bilinear", plot_aligned=False,
)
torch.cuda.synchronize()
print(f"Alignment: {perf_counter() - started:.2f} s")

## Inspect before saving

Merge a small region with the complete detector and float32 intensities. All seven inputs remain available for another region or the full export below. Coordinates are `(row_start, row_stop, column_start, column_stop)`, with exclusive stops.


In [ ]:
started = perf_counter()
patch = maped.merge_datasets(
    scan_region=(252, 260, 252, 260), plot_result=False,
)
torch.cuda.synchronize()
print(f"Selected region: {perf_counter() - started:.3f} s")
patch_viewer = maped.show()
patch_viewer


## Keep the complete merged result on the GPU

Choose `dtype="scaled_uint16"`. Calibration, region sizes and packing are automatic. MAPED releases its owned inputs after the complete result is resident. All scientific merging remains float32.


In [ ]:
started = perf_counter()
merged = maped.merge_datasets(dtype="scaled_uint16", plot_result=False)
torch.cuda.synchronize()
print(f"Merge + packed output: {perf_counter() - started:.2f} s")


In [ ]:
viewer = maped.show()
viewer

Optional saving uses the existing resident, without another merge or precision conversion:

```python
from quantem.gpu import io
io.save(OUTPUT, merged)
```

Full merging releases owned inputs; the packed output remains usable. Close viewers before releasing the remaining resources:

```python
patch_viewer.close()
viewer.close()
maped.close()
```


## Measured workflow

For these seven acquisitions, merge/conversion/packing/summaries took **6.19–10.34 s on CUDA** and **11.79–12.26 s on Torch MPS**. Loading is separate: **20.59–24.14 s CUDA**, **10.35–10.74 s MPS** in these sessions. Sampled peaks remained below 24 GiB on the larger test hosts; this is not physical 24 GB laptop qualification. See [timings and numerical qualification](../../docs/development/maped-scaled-storage-performance.md) for measurement boundaries.

![Three diffraction patterns from the packed merged output](selected_diffraction_patterns.png)


### Lower-memory saved merge

On a smaller Mac, use `save_to="merged_master.h5"` in the merge call above. It streams calibrated output regions to disk, releases MAPED-owned input tilts, and reopens the complete packed result for `maped.show()`. The no-file call uses more memory. The saved workflow measured 11.89 GiB on a larger MPS machine; physical 16 GB Mac and browser qualification is still pending. See `docs/development/maped-16gb-memory.md`.